# 07 — Multicollinearity Analizi (Hafta 9)

7 bağımsız değişken arasında çoklu doğrusal bağlantı (VIF) ve pairwise
korelasyon redundancy'si değerlendirilir.

**Eşikler:**
- VIF > 10 → ciddi multicollinearity, eleme gerekli
- |r| > 0.7 → güçlü pairwise korelasyon

**Önkoşul:** `grid_30m_standardized.gpkg` (Hafta 8).

**Çıktı:** Karar notu (`tables/07_vif_analysis.csv`) + final modelleme seti tanımı.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    FIGURES, TABLES, GRID_30M_STANDARDIZED, GRID_30M_MODELING,
    FEATURE_COLUMNS, TARGET_COLUMN, VIF_THRESHOLD, SELECTED_FEATURES,
)
from src.feature_selection import (
    compute_vif, iterative_vif_drop, correlation_redundancy,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Yükle ---
grid = gpd.read_file(GRID_30M_STANDARDIZED)
z_cols = [f"{c}_z" for c in FEATURE_COLUMNS]
print(f"Grid: {len(grid):,} hücre")
print(f"Z-skor kolonları ({len(z_cols)}): {z_cols}")

In [ ]:
# --- Korelasyon redundancy ---
for thr in [0.7, 0.5, 0.3]:
    red = correlation_redundancy(grid, z_cols, threshold=thr)
    print(f"|r| > {thr}: {len(red)} çift")
    if not red.empty and thr <= 0.5:
        print(red.to_string(index=False))
        print()

In [ ]:
# --- Tam korelasyon matrisi ---
fig, ax = plt.subplots(figsize=(9, 7))
z_corr = grid[z_cols + [TARGET_COLUMN]].corr().round(3)
sns.heatmap(z_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, cbar_kws={"label": "Pearson r"}, vmin=-1, vmax=1,
            annot_kws={"size": 9})
ax.set_title("7 z-skor değişken + LST korelasyon matrisi")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "07_correlation_matrix.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- VIF: 7 z-skor değişkenin tümü ---
vif = compute_vif(grid, z_cols, fillna_with=0.0)
print("VIF değerleri (azalan):")
print(vif.round(3).to_string())
print(f"\nEşik: {VIF_THRESHOLD}")
above = vif[vif > VIF_THRESHOLD].index.tolist()
print(f"Eşik üstü: {above if above else 'YOK — multicollinearity tespit edilmedi'}")

In [ ]:
# --- Iteratif VIF drop (multicollinearity olsaydı ne yapardık) ---
final_cols, history = iterative_vif_drop(
    grid, z_cols, threshold=VIF_THRESHOLD, fillna_with=0.0,
)
print(f"İteratif drop sonrası kalan ({len(final_cols)}):")
for c in final_cols:
    print(f"  {c}")
print(f"\nElenen: {set(z_cols) - set(final_cols) or 'yok'}")
print(f"\nİterasyon sayısı: {history.shape[1]}")

In [ ]:
# --- VIF görselleştirme ---
fig, ax = plt.subplots(figsize=(10, 6))
vif_sorted = vif.sort_values()
colors = ["#2A9D8F" if v < 5 else "#E9C46A" if v < VIF_THRESHOLD else "#E76F51"
          for v in vif_sorted.values]
ax.barh(range(len(vif_sorted)), vif_sorted.values, color=colors, edgecolor="black")
ax.set_yticks(range(len(vif_sorted)))
ax.set_yticklabels(vif_sorted.index, fontsize=9)
ax.axvline(VIF_THRESHOLD, color="red", linestyle="--", linewidth=1.5,
           label=f"Eşik (VIF={VIF_THRESHOLD})")
ax.axvline(5, color="orange", linestyle=":", linewidth=1, label="Dikkat (VIF=5)")
for i, v in enumerate(vif_sorted.values):
    ax.text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8)
ax.set_xlabel("VIF")
ax.set_title("Variance Inflation Factor — 7 değişken")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "07_vif_bars.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Pearson vs Spearman karşılaştırması (LST ile) ---
rows = []
for c in FEATURE_COLUMNS:
    r_pearson_raw = grid[[c, TARGET_COLUMN]].corr().iloc[0, 1]
    r_pearson_z   = grid[[f"{c}_z", TARGET_COLUMN]].corr().iloc[0, 1]
    r_spearman    = grid[[c, TARGET_COLUMN]].corr(method="spearman").iloc[0, 1]
    rows.append({
        "degisken": c,
        "r_pearson_raw": round(float(r_pearson_raw), 3),
        "r_pearson_z":   round(float(r_pearson_z), 3),
        "r_spearman":    round(float(r_spearman), 3),
        "VIF": round(float(vif[f"{c}_z"]), 3),
    })
comp = pd.DataFrame(rows).sort_values("VIF", ascending=False)
print("LST ile korelasyon + VIF karşılaştırması:")
print(comp.to_string(index=False))

## Karar Notu

### VIF perspektifi: multicollinearity YOK

Tüm 7 değişkenin VIF değeri **1.0-1.8 arasında**. Eşik 10'un çok altında, 5'in bile altında. Pairwise korelasyon |r| > 0.7 olan çift yok. Yani değişkenler birbirinden **bağımsız bilgi** taşıyor.

### Promptun "7 → 5" hedefi

Orijinal proje hipotezi VIF analizi sonrası 5 değişkene inmekti. Ancak VIF temizken parsimony adına 2 değişken atılırsa **bilgi kaybı** olur — özellikle:
- `building_density_per_km2` (Pearson 0.10, ama yapı yoğunluğu farklı bilgi)
- `ndvi_mean` (Spearman -0.25, monotonik non-lineer ilişki, RF'in yakalaması beklenen)

### Karar: 7 değişkeni de modele sok

- **Modelleme seti:** 7 değişkenin tümü (`SELECTED_FEATURES = FEATURE_COLUMNS`)
- Random Forest non-lineer ilişkileri yakalayabilir; permutation importance Hafta 10'da hangilerinin gerçekten katkı sağladığını gösterecek.
- Tez metni: VIF analizi multicollinearity göstermediği için 7'nin tümü tutuldu.

### DTC_breeze özel notu (Hafta 8'den devam)

1066 saturated hücre Pearson r=-0.654 sinyalini şişiriyor; log+z sonrası r=-0.17, Spearman=0.02. RF non-lineer olduğundan saturated outlier'ları doğal şekilde işleyecek; raw `dtc_breeze_m` modelin girdisi olacak.

## Sonraki Adım

`08_rf_model.ipynb` (Hafta 10) — Random Forest eğitimi + 5 katmanlı validation (rastgele CV, mekânsal CV, hold-out, yıllar arası, permutation null).

In [ ]:
# --- Modelleme tablosunu kaydet ---
# Tüm 7 raw + 7 z-skor + LST. Hafta 10 RF için hazır.
raw_cols = list(FEATURE_COLUMNS)
log_cols = [c for c in grid.columns if c.endswith("_log")]
z_cols2 = [f"{c}_z" for c in FEATURE_COLUMNS]
id_cols = [c for c in ["cell_id", "i", "j", "parent_id"] if c in grid.columns]

keep = id_cols + raw_cols + log_cols + z_cols2 + [TARGET_COLUMN, "geometry"]
modeling = grid[keep].copy()
modeling.to_file(GRID_30M_MODELING, driver="GPKG")
print(f"Modelleme tablosu: {GRID_30M_MODELING}")
print(f"Boyut: {GRID_30M_MODELING.stat().st_size/1024/1024:.2f} MB")
print(f"Kolonlar ({len(modeling.columns)}): "
      f"{len(id_cols)} id + {len(raw_cols)} raw + {len(log_cols)} log + {len(z_cols2)} z + LST + geometry")

# Summary CSV
comp.to_csv(TABLES / "07_vif_analysis.csv", index=False, encoding="utf-8-sig")
print(f"\nVIF analizi: {TABLES / '07_vif_analysis.csv'}")